In [1]:
import piplite
await piplite.install(['ipywidgets', 'matplotlib'])

In [6]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox
from IPython.display import display, clear_output, HTML

# =====================================================================
# Função que gera o gráfico com limites adaptativos e vetores limpos
# =====================================================================
def make_corner_plot(beta_deg, show_field=True,
                     show_equipotentials=True, show_walls=True):
    beta = np.deg2rad(beta_deg)
    alpha = np.pi / beta
    expoente = alpha - 1
    r_max = 1.4

    # Identificação exata do ponto crítico em 180°
    is_flat = abs(beta_deg - 180.0) < 0.2

    if beta_deg < 180.0 and not is_flat:
        cor = "#1f77b4"  # Canto convexo
    elif is_flat:
        cor = "#2ca02c"  # Superfície plana (exatamente 180°)
    else:
        cor = "#d62728"  # Canto côncavo

    fig, ax = plt.subplots(figsize=(7.5, 6.5), layout='constrained')

    # -----------------------------------------------------------------
    # 1. Limites Dinâmicos dos Eixos (Evita cortes)
    # -----------------------------------------------------------------
    if beta < np.pi / 2:
        min_x = 0.0
    elif beta <= np.pi:
        min_x = r_max * np.cos(beta)
    else:
        min_x = -r_max
    max_x = r_max

    if beta <= np.pi:
        min_y = 0.0
    elif beta <= 3 * np.pi / 2:
        min_y = r_max * np.sin(beta)
    else:
        min_y = -r_max

    if beta <= np.pi / 2:
        max_y = r_max * np.sin(beta)
    else:
        max_y = r_max

    pad = 0.15
    ax.set_xlim(min_x - pad, max_x + pad)
    ax.set_ylim(min_y - pad, max_y + pad)

    # -----------------------------------------------------------------
    # 2. Paredes Condutoras
    # -----------------------------------------------------------------
    if show_walls:
        ax.plot([0, r_max], [0, 0], 'k-', linewidth=4, zorder=5)
        ax.plot([0, r_max * np.cos(beta)], [0, r_max * np.sin(beta)],
                'k-', linewidth=4, zorder=5)

    # -----------------------------------------------------------------
    # 3. Linhas Equipotenciais
    # -----------------------------------------------------------------
    if show_equipotentials:
        if is_flat:
            xc = np.linspace(-1.3, 1.3, 120)
            yc = np.linspace(0.001, 1.3, 80)
            Xc, Yc = np.meshgrid(xc, yc)
            U = Yc
        else:
            rho_c = np.linspace(0.001, 1.3, 160)
            phi_c = np.linspace(0, beta, 160)
            Rc, Phic = np.meshgrid(rho_c, phi_c)
            U = Rc**alpha * np.sin(alpha * Phic)
            Xc = Rc * np.cos(Phic)
            Yc = Rc * np.sin(Phic)

        ax.contour(Xc, Yc, U, levels=10, colors='gray',
                   linewidths=0.7, alpha=0.55, zorder=2)

    # -----------------------------------------------------------------
    # 4. Campo Vetorial (Densidade Otimizada sem Poluição Visual)
    # -----------------------------------------------------------------
    if show_field:
        if is_flat:
            # Grade cartesiana limpa para o caso de placas paralelas
            x_grid = np.linspace(-1.2, 1.2, 15)
            y_grid = np.linspace(0.12, 1.2, 7)
            X, Y = np.meshgrid(x_grid, y_grid)
            Ex = np.zeros_like(X)
            Ey = -np.ones_like(Y)
            mag = np.ones_like(X)
        else:
            # Controle de densidade: reduz o acúmulo de setas perto do vértice
            n_rho = 10
            n_phi = int(np.clip(beta_deg / 22, 6, 16))
            
            # rho começa em 0.12 para não sobrepor setas no centro
            rho = np.linspace(0.12, 1.22, n_rho)
            phi = np.linspace(0.03 * beta, 0.97 * beta, n_phi)
            R, Phi = np.meshgrid(rho, phi)
            
            E_rho = -alpha * R**(alpha - 1) * np.sin(alpha * Phi)
            E_phi = -alpha * R**(alpha - 1) * np.cos(alpha * Phi)
            
            X = R * np.cos(Phi)
            Y = R * np.sin(Phi)
            Ex = E_rho * np.cos(Phi) - E_phi * np.sin(Phi)
            Ey = E_rho * np.sin(Phi) + E_phi * np.cos(Phi)
            mag = np.sqrt(Ex**2 + Ey**2)

        mag_safe = np.where(mag == 0, 1, mag)
        q = ax.quiver(X, Y, Ex / mag_safe, Ey / mag_safe, mag,
                      cmap='plasma', scale=22, width=0.005,
                      pivot='mid', alpha=0.95, zorder=3)
        cbar = plt.colorbar(q, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(r'$|\vec{E}|$ (u.a.)', fontsize=11)

    ax.set_aspect('equal')
    ax.set_xlabel(r'$x$', fontsize=12)
    ax.set_ylabel(r'$y$', fontsize=12)
    ax.grid(alpha=0.15, linestyle=':')
    ax.set_title(
        rf'$\beta = {beta_deg:.1f}^\circ \quad | \quad '
        rf'\pi/\beta - 1 = {expoente:.3f}$',
        fontsize=13, color=cor, pad=10
    )
    plt.show()
    plt.close(fig)

# =====================================================================
# Descrição dinâmica do regime físico
# =====================================================================
def describe_regime(beta_deg):
    beta = np.deg2rad(beta_deg)
    alpha = np.pi / beta
    expoente = alpha - 1
    is_flat = abs(beta_deg - 180.0) < 0.2

    if beta_deg < 180.0 and not is_flat:
        cor = "#1f77b4"
        regime = "Canto Convexo"
        texto = (
            f"O expoente <b>&pi;/&beta; &minus; 1 = {expoente:.3f}</b> é <b>positivo</b>. "
            "O campo elétrico e a densidade de carga <b>tendem a zero</b> quando "
            "&rho; &rarr; 0. As linhas de campo se <b>espalham</b> ao se aproximarem "
            "do vértice, pois o ângulo disponível é menor que &pi;."
        )
    elif is_flat:
        cor = "#2ca02c"
        regime = "Superfície Plana (Ponto Crítico)"
        texto = (
            f"O expoente <b>&pi;/&beta; &minus; 1 = {expoente:.3f}</b> é <b>nulo</b>. "
            "O campo é <b>uniforme</b> em toda a região, formando linhas de campo verticais "
            "e perfeitamente paralelas sobre a placa condutora, recuperando o caso do "
            "capacitor de placas paralelas."
        )
    else:
        cor = "#d62728"
        regime = "Canto Côncavo"
        texto = (
            f"O expoente <b>&pi;/&beta; &minus; 1 = {expoente:.3f}</b> é <b>negativo</b>. "
            "O campo elétrico e a densidade de carga <b>divergem</b> quando "
            "&rho; &rarr; 0. As linhas de campo se <b>concentram</b> no vértice, "
            "caracterizando o <b>efeito para-raios</b>."
        )

    return f"""
    <div style='border-left: 5px solid {cor}; padding: 12px;
                margin: 10px 0; background: #f8f8f8; border-radius: 4px;'>
      <b style='color: {cor}; font-size: 14px;'>Regime: {regime}</b><br>
      <span style='font-size: 13px; line-height: 1.6;'>{texto}</span>
    </div>
    """

# =====================================================================
# Função controladora interativa
# =====================================================================
def explorar(beta_deg, show_field, show_equipotentials, show_walls):
    clear_output(wait=True)
    make_corner_plot(beta_deg, show_field, show_equipotentials, show_walls)
    display(HTML(describe_regime(beta_deg)))

interact(
    explorar,
    beta_deg=FloatSlider(min=20, max=359, step=1, value=90,
                         description='β (graus):',
                         style={'description_width': 'initial'},
                         continuous_update=False),
    show_field=Checkbox(value=True, description='Campo vetorial'),
    show_equipotentials=Checkbox(value=True, description='Equipotenciais'),
    show_walls=Checkbox(value=True, description='Paredes condutoras')
)

interactive(children=(FloatSlider(value=90.0, continuous_update=False, description='β (graus):', max=359.0, mi…

<function __main__.explorar(beta_deg, show_field, show_equipotentials, show_walls)>